Fraud Detection & Risk Scoring


Project summary
A risk-scoring model for Finlora's transaction monitoring, trained on transaction and account behavior to assign every payment a fraud probability rather than a pass/fail rule. It benchmarks a Logistic Regression baseline against a Random Forest classifier on 126,000 transactions across 7,200 accounts, built on Finlora's own transaction and account data.

Business context
Finlora's current monitoring relies on fixed-threshold rules that worked when volume was low, but fraud is now subtler and volume is growing across individual and business accounts, KYC tiers, channels, and merchant categories. That's leaving two problems at once: confirmed fraud slipping through, and legitimate payroll, wire, and P2P transfers getting blocked or delayed. Analysts also have no way to prioritize alerts every flag gets equal urgency regardless of actual risk.

Project purpose
Replace the binary rule trigger with a continuous, explainable score built from account-level behavioral features spend baseline, transaction velocity, device novelty, and cross-border activity and identify which of these signals most strongly separate fraud from legitimate activity, informing both the model and any rules that remain.

Expected outcomes
A validated model that catches materially more true fraud than the current rule-based baseline, at a false-positive rate the risk team can operationally sustain. Every flag comes with visible contributing signals rather than a black-box decision, feeding a prioritized analyst review queue, with precision/recall/F1/ROC-AUC evaluation and tunable thresholds at 70/80/90% recall.

In [3]:
# Import the libraries and read the data

In [4]:
import numpy as np
import pandas as pandas
import seaborn as sns
import matplotlib.pyplot as plt


In [5]:
import pandas as pd


df1=pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\AMDARI\Finlora Finance\data\raw data\finlora_accounts.csv")

In [6]:
df2=pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\AMDARI\Finlora Finance\data\raw data\finlora_transactions.csv")

In [7]:
df=pd.merge(df1,df2,how='outer')

In [8]:
df.head()

,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd,transaction_id,timestamp,...,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250216186265,2025-02-16 14:55:35,...,9.21,9143.33,0.0,NG,0.0,NaN,NaN,903.0,Completed,0.0
1,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250423143305,2025-04-23 13:35:40,...,1.00,9143.33,0.0,NG,0.0,NaN,NaN,969.0,Declined,0.0
2,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250424154897,2025-04-24 12:30:34,...,2.26,9143.33,0.0,NG,0.0,DEV-9055235108,0.0,970.0,Completed,0.0
3,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250428101772,2025-04-28 14:41:59,...,0.21,14891.16,0.0,NG,0.0,NaN,NaN,974.0,Completed,0.0
4,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250628102827,2025-06-28 16:15:47,...,1.97,49609.59,0.0,NG,0.0,DEV-9055235108,0.0,1035.0,Declined,0.0


In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126056 entries, 0 to 126055
Data columns (total 27 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   account_id                   126056 non-null  object 
 1   account_holder_name          126056 non-null  object 
 2   account_type                 126056 non-null  object 
 3   home_country                 126056 non-null  object 
 4   currency                     126056 non-null  object 
 5   kyc_tier                     126056 non-null  object 
 6   account_created_date         126056 non-null  object 
 7   personal_spend_baseline_usd  126056 non-null  float64
 8   transaction_id               126000 non-null  object 
 9   timestamp                    126000 non-null  object 
 10  day_of_week                  126000 non-null  object 
 11  hour_of_day                  126000 non-null  float64
 12  description                  126000 non-null  object 
 13 

In [10]:
# investigate missing data
df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                    56
timestamp                         56
day_of_week                       56
hour_of_day                       56
description                       56
merchant_name                  28666
merchant_category                 56
channel                           56
amount                            56
amount_to_avg_ratio               56
avg_transaction_amount_30d        56
transaction_velocity_1h           56
transaction_country               56
is_cross_border                   56
device_id                      14390
is_new_device                  14390
account_age_days                  56
status                            56
is_fraud                          56
d

In [11]:
# merging using common columns

In [12]:
common_cols=['account_id','account_type','home_country','currency','kyc_tier']

In [13]:
merged_df=pd.merge(df1,df2,on=common_cols, how='outer')

In [14]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126056 entries, 0 to 126055
Data columns (total 27 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   account_id                   126056 non-null  object 
 1   account_holder_name          126056 non-null  object 
 2   account_type                 126056 non-null  object 
 3   home_country                 126056 non-null  object 
 4   currency                     126056 non-null  object 
 5   kyc_tier                     126056 non-null  object 
 6   account_created_date         126056 non-null  object 
 7   personal_spend_baseline_usd  126056 non-null  float64
 8   transaction_id               126000 non-null  object 
 9   timestamp                    126000 non-null  object 
 10  day_of_week                  126000 non-null  object 
 11  hour_of_day                  126000 non-null  float64
 12  description                  126000 non-null  object 
 13 

In [15]:
merged_df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                    56
timestamp                         56
day_of_week                       56
hour_of_day                       56
description                       56
merchant_name                  28666
merchant_category                 56
channel                           56
amount                            56
amount_to_avg_ratio               56
avg_transaction_amount_30d        56
transaction_velocity_1h           56
transaction_country               56
is_cross_border                   56
device_id                      14390
is_new_device                  14390
account_age_days                  56
status                            56
is_fraud                          56
d

In [16]:
#there is no difference!!

In [17]:
#drop missing values,56 missing point values
merged_df.dropna(subset=['transaction_id'])

,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd,transaction_id,timestamp,...,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250216186265,2025-02-16 14:55:35,...,9.21,9143.33,0.0,NG,0.0,NaN,NaN,903.0,Completed,0.0
1,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250423143305,2025-04-23 13:35:40,...,1.00,9143.33,0.0,NG,0.0,NaN,NaN,969.0,Declined,0.0
2,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250424154897,2025-04-24 12:30:34,...,2.26,9143.33,0.0,NG,0.0,DEV-9055235108,0.0,970.0,Completed,0.0
3,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250428101772,2025-04-28 14:41:59,...,0.21,14891.16,0.0,NG,0.0,NaN,NaN,974.0,Completed,0.0
4,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250628102827,2025-06-28 16:15:47,...,1.97,49609.59,0.0,NG,0.0,DEV-9055235108,0.0,1035.0,Declined,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126051,FLR-ACC-107199,Peter Taylor,Individual,FR,EUR,Tier2_Verified,2024-04-15,153.79,FLR260619214696,2026-06-19 06:06:56,...,2.01,99.26,0.0,FR,0.0,DEV-1194729205,1.0,795.0,Completed,0.0
126052,FLR-ACC-107199,Peter Taylor,Individual,FR,EUR,Tier2_Verified,2024-04-15,153.79,FLR260620105850,2026-06-20 18:39:55,...,0.98,149.63,0.0,FR,0.0,DEV-1904412328,0.0,796.0,Completed,0.0
126053,FLR-ACC-107199,Peter Taylor,Individual,FR,EUR,Tier2_Verified,2024-04-15,153.79,FLR260628219315,2026-06-28 21:05:20,...,0.04,173.00,0.0,FR,0.0,DEV-1143115776,0.0,804.0,Completed,0.0
126054,FLR-ACC-107199,Peter Taylor,Individual,FR,EUR,Tier2_Verified,2024-04-15,153.79,FLR260715110692,2026-07-15 04:44:38,...,0.16,117.89,0.0,FR,0.0,DEV-1904412328,0.0,821.0,Completed,0.0


In [18]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126056 entries, 0 to 126055
Data columns (total 27 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   account_id                   126056 non-null  object 
 1   account_holder_name          126056 non-null  object 
 2   account_type                 126056 non-null  object 
 3   home_country                 126056 non-null  object 
 4   currency                     126056 non-null  object 
 5   kyc_tier                     126056 non-null  object 
 6   account_created_date         126056 non-null  object 
 7   personal_spend_baseline_usd  126056 non-null  float64
 8   transaction_id               126000 non-null  object 
 9   timestamp                    126000 non-null  object 
 10  day_of_week                  126000 non-null  object 
 11  hour_of_day                  126000 non-null  float64
 12  description                  126000 non-null  object 
 13 

In [19]:
merged_df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                    56
timestamp                         56
day_of_week                       56
hour_of_day                       56
description                       56
merchant_name                  28666
merchant_category                 56
channel                           56
amount                            56
amount_to_avg_ratio               56
avg_transaction_amount_30d        56
transaction_velocity_1h           56
transaction_country               56
is_cross_border                   56
device_id                      14390
is_new_device                  14390
account_age_days                  56
status                            56
is_fraud                          56
d

In [20]:
# Investigate missing values
#drop the 56 data points

merged_df = df.dropna(subset=['transaction_id'])

In [21]:
merged_df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                     0
timestamp                          0
day_of_week                        0
hour_of_day                        0
description                        0
merchant_name                  28610
merchant_category                  0
channel                            0
amount                             0
amount_to_avg_ratio                0
avg_transaction_amount_30d         0
transaction_velocity_1h            0
transaction_country                0
is_cross_border                    0
device_id                      14334
is_new_device                  14334
account_age_days                   0
status                             0
is_fraud                           0
d

In [22]:
df=merged_df


In [23]:
df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                     0
timestamp                          0
day_of_week                        0
hour_of_day                        0
description                        0
merchant_name                  28610
merchant_category                  0
channel                            0
amount                             0
amount_to_avg_ratio                0
avg_transaction_amount_30d         0
transaction_velocity_1h            0
transaction_country                0
is_cross_border                    0
device_id                      14334
is_new_device                  14334
account_age_days                   0
status                             0
is_fraud                           0
d

In [24]:
# for merchant name, 

#1: Standardize the text ---
# This converts the word "TO" into a hyphen " - " so all rows look the same
df["Standardized_Description"] = df["description"].str.replace(
       r"\s+TO\s+", " - ", regex=True, case=False)

# 2: Split and extract the fallback names ---

# Split by the hyphen, take the right side (index 1), and clean it up
fallback_names = df["Standardized_Description"].str.split(" - ", n=1, expand=True)[1]

# Strip accidental spaces and convert to Title Case
df["Fallback_Merchant"] = fallback_names.str.strip().str.title()


# 3: Merge columns and leave unresolved spots blank ---

# Combine original column and fallback column only (do not apply 'no_merchant' yet)
df["Cleaned_Merchant"] = df["merchant_name"].fillna(df["Fallback_Merchant"])


# 4: Identify non-merchant keywords and overwrite them ---

# Define the list of bank system terms to flag
non_merchant_keywords = "NIP TRANSFER|SALARY"

# 2. Check which rows contain any of these keywords (ignoring upper/lower case)
is_non_merchant = df["description"].str.contains(non_merchant_keywords, case=False, na=False)

# 3. For any matching row, overwrite the Cleaned_Merchant column to 'no_merchant'
df.loc[is_non_merchant, "Cleaned_Merchant"] = "no_merchant"

# Show the results after filtering
print(df[["description", "Cleaned_Merchant"]])


                                  description      Cleaned_Merchant
0                         CNP PURCHASE - BOLT                  Bolt
1                        WEB PURCHASE - SLACK                 Slack
2       MOBILE PURCHASE - JUSTRITE SUPERSTORE   Justrite Superstore
3         CNP PURCHASE - ADOBE CREATIVE CLOUD  Adobe Creative Cloud
4               MOBILE PURCHASE - APPLE STORE           Apple Store
...                                       ...                   ...
126051             MOBILE PURCHASE - BEST BUY              Best Buy
126052          MOBILE PURCHASE - AXA MANSARD           AXA Mansard
126053                    CNP PURCHASE - GOTV                  GOtv
126054              API PURCHASE - JUMIA MART            Jumia Mart
126055      WEB PURCHASE - APPLE STORE ONLINE    Apple Store Online

[126000 rows x 2 columns]


In [25]:
df['Cleaned_Merchant'].isnull().sum()

np.int64(0)

In [26]:
df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                     0
timestamp                          0
day_of_week                        0
hour_of_day                        0
description                        0
merchant_name                  28610
merchant_category                  0
channel                            0
amount                             0
amount_to_avg_ratio                0
avg_transaction_amount_30d         0
transaction_velocity_1h            0
transaction_country                0
is_cross_border                    0
device_id                      14334
is_new_device                  14334
account_age_days                   0
status                             0
is_fraud                           0
S

In [27]:
# Drop the original messy column completely

df = df.drop(columns=["merchant_name"])

# Insert clean column into the 14th slot (Index 13)

df.insert(13, "merchant_name", df.pop("Cleaned_Merchant"))

# Show the columns to verify their new order
print(df.columns.tolist())

['account_id', 'account_holder_name', 'account_type', 'home_country', 'currency', 'kyc_tier', 'account_created_date', 'personal_spend_baseline_usd', 'transaction_id', 'timestamp', 'day_of_week', 'hour_of_day', 'description', 'merchant_name', 'merchant_category', 'channel', 'amount', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'transaction_velocity_1h', 'transaction_country', 'is_cross_border', 'device_id', 'is_new_device', 'account_age_days', 'status', 'is_fraud', 'Standardized_Description', 'Fallback_Merchant']


In [28]:
df.isnull().sum()

account_id                         0
account_holder_name                0
account_type                       0
home_country                       0
currency                           0
kyc_tier                           0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                     0
timestamp                          0
day_of_week                        0
hour_of_day                        0
description                        0
merchant_name                      0
merchant_category                  0
channel                            0
amount                             0
amount_to_avg_ratio                0
avg_transaction_amount_30d         0
transaction_velocity_1h            0
transaction_country                0
is_cross_border                    0
device_id                      14334
is_new_device                  14334
account_age_days                   0
status                             0
is_fraud                           0
S

In [29]:
#For device_id, fill with placceholder 'missing_device'

df['device_id']=df['device_id'].fillna('missing_device')

In [30]:
df['device_id'].isnull().sum()

np.int64(0)

In [31]:
#For is_new_device, fill blanks with (-1)

# Standardise blanks and fill with -1
df["is_new_device"] = (df["is_new_device"].astype(str).str.strip().replace(
    ["", "None", "nan"], np.nan))

df["is_new_device"] = df["is_new_device"].fillna(-1)

df['is_new_device'].isnull().sum()

np.int64(0)

In [32]:
df.isnull().sum()

account_id                     0
account_holder_name            0
account_type                   0
home_country                   0
currency                       0
kyc_tier                       0
account_created_date           0
personal_spend_baseline_usd    0
transaction_id                 0
timestamp                      0
day_of_week                    0
hour_of_day                    0
description                    0
merchant_name                  0
merchant_category              0
channel                        0
amount                         0
amount_to_avg_ratio            0
avg_transaction_amount_30d     0
transaction_velocity_1h        0
transaction_country            0
is_cross_border                0
device_id                      0
is_new_device                  0
account_age_days               0
status                         0
is_fraud                       0
Standardized_Description       0
Fallback_Merchant              0
dtype: int64

In [33]:
#change data types: account_created_date and timestamp, to datetime

df['account_created_date']=pd.to_datetime(df['account_created_date'])

df['timestamp']=pd.to_datetime(df['timestamp'])

In [34]:
# extract month and year from account_created_date

df['year'] = df['account_created_date'].dt.year
df['month'] = df['account_created_date'].dt.month_name()
df['day'] = df['account_created_date'].dt.day

In [35]:
print(df.columns)

Index(['account_id', 'account_holder_name', 'account_type', 'home_country',
       'currency', 'kyc_tier', 'account_created_date',
       'personal_spend_baseline_usd', 'transaction_id', 'timestamp',
       'day_of_week', 'hour_of_day', 'description', 'merchant_name',
       'merchant_category', 'channel', 'amount', 'amount_to_avg_ratio',
       'avg_transaction_amount_30d', 'transaction_velocity_1h',
       'transaction_country', 'is_cross_border', 'device_id', 'is_new_device',
       'account_age_days', 'status', 'is_fraud', 'Standardized_Description',
       'Fallback_Merchant', 'year', 'month', 'day'],
      dtype='object')


In [36]:
# normalise the currency


# Define exchange rates relative to base currency (Target Base = GBP)

exchange_rates_to_GBP = {
    'GBP': 1.0,         # Base currency
    'NGN': 0.00056,     # 1 NGN =  0.00056 GBP
    'USD': 0.77,        # 1 USD = 0.77 GBP
    'EUR': 0.84         # 1 EUR = 0.84 GBP
    }

#Map the rates to the DataFrame and calculate the normalized amount

df['exchange_rate'] = df['currency'].map(exchange_rates_to_GBP)

df['amount_in_GBP'] = df['amount'] * df['exchange_rate']

df.head(2)


,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd,transaction_id,timestamp,...,account_age_days,status,is_fraud,Standardized_Description,Fallback_Merchant,year,month,day,exchange_rate,amount_in_GBP
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250216186265,2025-02-16 14:55:35,...,903.0,Completed,0.0,CNP PURCHASE - BOLT,Bolt,2022,August,28,0.00056,47.170480
1,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23,FLR250423143305,2025-04-23 13:35:40,...,969.0,Declined,0.0,WEB PURCHASE - SLACK,Slack,2022,August,28,0.00056,5.120265


In [37]:
#Check data types:

#convert hour_of_the_day, is_cross_border, account_age_days, is_fraud to int data type

int_columns = ['hour_of_day', 'is_cross_border', 'is_new_device', 'account_age_days', 'is_fraud']


for col in int_columns:
    
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')


In [38]:
# convert month to data type object

df['month'] = df['account_created_date'].dt.month_name()

In [39]:
print(df.dtypes)

account_id                             object
account_holder_name                    object
account_type                           object
home_country                           object
currency                               object
kyc_tier                               object
account_created_date           datetime64[ns]
personal_spend_baseline_usd           float64
transaction_id                         object
timestamp                      datetime64[ns]
day_of_week                            object
hour_of_day                             Int64
description                            object
merchant_name                          object
merchant_category                      object
channel                                object
amount                                float64
amount_to_avg_ratio                   float64
avg_transaction_amount_30d            float64
transaction_velocity_1h               float64
transaction_country                    object
is_cross_border                   

In [40]:
# investigate duplicates

df.duplicated().sum()

np.int64(0)

In [45]:
# no duplicates found

In [44]:
# save DataFrame 

df.to_csv('cleaned_data.csv', index=False)


Observations
- A new column was added for ammount in USD 

- Correction of account age days
when the account age days was corrected, many values were 
negative. when researched, it was observed that, for 
machine learning, these values have to be floored to zeros. 
that was the reason we had zeros in the first place. the data was left unchanged and a new column was added for is same day created accounts (ie the zeros) 

In [ ]:
#updated cleaned data file path
# ("C:\Users\hp\OneDrive\Desktop\AMDARI\Finlora Finance\notebook\cleaned_data2.csv")